<a href="https://colab.research.google.com/github/AnnaZapototska/colab-code-work/blob/development/hw_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3

Homework for the module "Learning Algorithms with a Teacher Part 3"# New Section

In [1]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 1.2 MB/s eta 0:00:00


## Part 1

In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import mean_absolute_percentage_error

## Part 2

In [3]:
train_url = "https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_train_data.csv"
valid_url = "https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_valid_data.csv"

train_data = pd.read_csv(train_url)
valid_data = pd.read_csv(valid_url)

print("Train dataset:")
display(train_data.head())

print("\nValidation dataset:")
display(valid_data.head())

Train dataset:


,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
0,Jennifer Hernandez,120-602-1220,3.0,Msc,Tier2,Mid,Yes,25/08/1972,98000
1,Timothy Walker,840-675-8650,5.0,PhD,Tier2,Senior,Yes,03/12/2013,135500
2,David Duran,556-293-8643,5.0,Msc,Tier2,Senior,Yes,19/07/2002,123500
3,Gloria Ortega,463-559-7474,3.0,Bsc,Tier3,Mid,No,19/02/1970,85000
4,Matthew Steele,968-091-7683,5.0,Bsc,Tier2,Senior,Yes,20/02/1970,111500



Validation dataset:


,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
0,Alvaro Johnson,320-636-8883,7,Bsc,Tier1,Senior,No,12/03/1978,109300
1,Austin Powers,903-121-1691,2,Msc,Tier1,Mid,Yes,13/03/1992,84800
2,Joshua Phil,673-972-2453,3,Bsc,Tier3,Mid,Yes,19/02/1988,98900
3,Mirinda Collins,310-364-6925,5,Msc,Tier2,Senior,No,20/03/1989,116500
4,Mustapha Green,401-249-3912,3,PhD,Tier1,Junior,Yes,21/03/1979,75800


In [4]:
print("Train shape:", train_data.shape)
print("Valid shape:", valid_data.shape)

Train shape: (249, 9)
Valid shape: (7, 9)


## Part 3

In [5]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           249 non-null    object 
 1   Phone_Number   249 non-null    object 
 2   Experience     247 non-null    float64
 3   Qualification  248 non-null    object 
 4   University     249 non-null    object 
 5   Role           246 non-null    object 
 6   Cert           247 non-null    object 
 7   Date_Of_Birth  249 non-null    object 
 8   Salary         249 non-null    int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 17.6+ KB


In [6]:
train_data.describe(include="all")

,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
count,249,249,247.000000,248,249,246,247,249,249.000000
unique,248,249,NaN,3,3,3,2,247,NaN
top,Eric Taylor,120-602-1220,NaN,Bsc,Tier1,Senior,Yes,25/08/1972,NaN
freq,2,1,NaN,113,94,128,129,2,NaN
mean,NaN,NaN,3.441296,NaN,NaN,NaN,NaN,NaN,98186.746988
std,NaN,NaN,1.496471,NaN,NaN,NaN,NaN,NaN,23502.622217
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,49500.000000
25%,NaN,NaN,2.000000,NaN,NaN,NaN,NaN,NaN,78500.000000
50%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,NaN,104500.000000
75%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN,NaN,116500.000000


In [7]:
train_data.isnull().sum()

,0
Name,0
Phone_Number,0
Experience,2
Qualification,1
University,0
Role,3
Cert,2
Date_Of_Birth,0
Salary,0


In [8]:
train_data.corr(numeric_only=True)["Salary"].sort_values(ascending=False)

,Salary
Salary,1.000000
Experience,0.814427


In [9]:
categorical_features = [
    "Qualification",
    "University",
    "Role",
    "Cert"
]

for feature in categorical_features:
    print(f"\n--- {feature} ---")
    print(
        train_data.groupby(feature)["Salary"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )


--- Qualification ---
               count           mean    median
Qualification                                
PhD               60  113083.333333  116250.0
Msc               75  102106.666667  112500.0
Bsc              113   87557.522124   88000.0

--- University ---
            count           mean    median
University                                
Tier2          93  101860.215054  106000.0
Tier3          62  100532.258065  109750.0
Tier1          94   93005.319149   98500.0

--- Role ---
        count           mean    median
Role                                  
Senior    128  116625.000000  116000.0
Mid        52   91076.923077   91500.0
Junior     66   67636.363636   66000.0

--- Cert ---
      count           mean    median
Cert                                
Yes     129  101360.465116  107500.0
No      118   94838.983051  100500.0


### EDA Summary
According to EDA, the most appropriate features for predicting Salary are Experience, Role, and Qualification. Experience has a strong positive relationship with salary (correlation 0.814), and for Role and Qualification, significant differences in average salary between categories are observed.

Name and Phone_Number have no meaningful value for prediction and will be removed. Date_Of_Birth contains anomalous values, so we will not use this feature either.

In [11]:
missing_data = train_data[
    train_data[
        ["Experience", "Qualification", "University", "Role", "Cert"]
    ].isnull().any(axis=1)
]

display(missing_data)

,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
26,Mr. Luis George,418-327-6776,5.0,NaN,Tier2,Senior,Yes,15/06/2001,111500
32,William Allen,505-962-3472,1.0,Msc,Tier3,NaN,Yes,18/01/1973,78500
43,Whitney Moore,294-513-7323,5.0,Msc,Tier1,NaN,No,19/10/1999,112500
48,Robert Wong,468-904-8098,5.0,Msc,Tier3,NaN,Yes,10/12/1972,129500
56,Melanie Smith,160-299-3920,5.0,Bsc,Tier1,Senior,NaN,11/10/1971,100500
62,Stephen Colon,401-249-3912,3.0,PhD,Tier1,Junior,NaN,21/03/1989,81500
72,Donald Perkins,708-475-1763,NaN,PhD,Tier2,Senior,No,12/04/1990,128500
76,Fernando Bryan,751-806-7172,NaN,Msc,Tier1,Junior,Yes,08/02/1988,68500


## Part 4

In [12]:
features = [
    "Experience",
    "Qualification",
    "University",
    "Role",
    "Cert"
]

X_train = train_data[features].copy()
y_train = train_data["Salary"].copy()

In [13]:
experience_median = X_train["Experience"].median()

X_train["Experience"] = X_train["Experience"].fillna(
    experience_median
)

In [14]:
for feature in [
    "Qualification",
    "University",
    "Role",
    "Cert"
]:
    X_train[feature] = X_train[feature].fillna(
        X_train[feature].mode()[0]
    )

In [15]:
X_train.isnull().sum()

,0
Experience,0
Qualification,0
University,0
Role,0
Cert,0


In [16]:
cat_cols = X_train.select_dtypes(include="object").columns

cat_cols

Index(['Qualification', 'University', 'Role', 'Cert'], dtype='object')

In [17]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [18]:
X_train_cat = encoder.fit_transform(X_train[cat_cols])

In [19]:
X_train = np.column_stack([X_train[["Experience"]].values, X_train_cat])

In [20]:
print(X_train.shape)

(249, 12)


In [23]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

## Part 5

In [34]:
model = KNeighborsRegressor(n_neighbors=15)

model.fit(X_train,y_train)

KNeighborsRegressor(n_neighbors=15)

## Part 6

In [35]:
X_valid = valid_data[features].copy()
y_valid = valid_data["Salary"].copy()

In [36]:
X_valid.isnull().sum()

,0
Experience,0
Qualification,0
University,0
Role,0
Cert,0


In [37]:
X_valid_cat = encoder.transform(X_valid[cat_cols])

In [38]:
X_valid = np.column_stack([X_valid[["Experience"]].values, X_valid_cat])

In [39]:
X_valid = scaler.transform(X_valid)

## Part 7

In [40]:
y_pred = model.predict(X_valid)

mape = mean_absolute_percentage_error(y_valid, y_pred)

print(f"Validation MAPE: {mape:.2%}")

Validation MAPE: 5.98%


In [42]:
comparison = pd.DataFrame({
    "Actual Salary": y_valid.values,
    "Predicted Salary": y_pred
})

display(comparison)

,Actual Salary,Predicted Salary
0,109300,108180.0
1,84800,92450.0
2,98900,90420.0
3,116500,113820.0
4,75800,88930.0
5,97300,97220.0
6,69800,81650.0


We are making expiriment with different k value.

In [41]:
for k in [10, 15, 20, 25, 50]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)

    mape = mean_absolute_percentage_error(y_valid, y_pred)

    print(f"k={k}: MAPE={mape:.2%}")

k=10: MAPE=7.30%
k=15: MAPE=5.98%
k=20: MAPE=7.80%
k=25: MAPE=8.38%
k=50: MAPE=7.90%


# Summary

In the course of the work, EDA was performed and a KNeighborsRegressor model was built for salary prediction. Features that have no practical value for prediction were removed, missing values ​​were processed, categorical features were encoded using OneHotEncoder, and the data was scaled using StandardScaler.

In the course of the work, several experiments were conducted with different numbers of features, data encoding and transformation methods, and different values ​​of k.

In particular, we tested:
*   the use of 3 and 5 features;
*   TargetEncoder and OneHotEncoder;
*   StandardScaler and PowerTransformer;
*   different values ​​of the parameter k in KNeighborsRegressor

The best result among the tested configurations was shown by the model with 5 features and k=15. The value of MAPE = 5.98% on the validation set is close to the expected result of 3–5%.

Thus experiments have shown that for kNN, proper feature training, scaling, and choosing the optimal value of k are particularly important.

